# NVIDIA NeMo + чистый YouTube-звук — выпуск 2

**Runtime → T4 GPU.** Меню Runtime → Change runtime type → T4 GPU.  
Потом **Restart session** и ячейки сверху вниз (Runtime → Run all).

Этот ноутбук: `https://youtu.be/h1615IhITQ4` (2 спикера).  
Скачивание: родной звук, **без** `--audio-format mp3`.  
Клипы из 44.1 kHz WAV. 16 kHz только для NeMo.

В конце скачай `nemo_speakers_native.zip` (Files слева) и пришли его сюда — обновлю сайт.

Репозитории: https://github.com/NVIDIA/NeMo · https://github.com/NVIDIA-NeMo/Speech

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "Runtime → T4 GPU, потом Restart session"
print("CUDA", torch.cuda.get_device_name(0))

In [ ]:
# Только установка. После этой ячейки: Runtime → Restart session.
# Потом эту ячейку больше не запускай — сразу nvidia-smi и скачивание.
!pip -q install -U yt-dlp soundfile omegaconf hydra-core "nemo_toolkit[asr]"
!pip -q install -U transformers huggingface_hub sentencepiece
print("OK. Теперь Runtime → Restart session, затем запусти ячейку с nvidia-smi и всё ниже.")

In [ ]:
# Выпуск 2. Родной звук, без перекодирования в MP3
URL = "https://youtu.be/h1615IhITQ4"

!rm -f native.* podcast_16k.wav podcast_hq.wav
!yt-dlp -f "bestaudio[acodec^=opus]/bestaudio/best" --no-playlist --restrict-filenames \
  --extractor-args "youtube:player_client=android,tv,web" \
  -o "native.%(ext)s" {URL}

!ls -la native.*
!ffprobe -hide_banner -show_entries stream=codec_name,sample_rate,bit_rate,channels -of compact native.*

!ffmpeg -y -i native.* -ac 1 -ar 16000 -c:a pcm_s16le podcast_16k.wav
!ffmpeg -y -i native.* -ac 1 -ar 44100 -c:a pcm_s16le podcast_hq.wav
!ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 podcast_16k.wav
!ffprobe -v error -show_entries stream=sample_rate,channels -of compact podcast_hq.wav

In [ ]:
import json
from pathlib import Path
from omegaconf import OmegaConf
from nemo.collections.asr.models import ClusteringDiarizer
import torch

!wget -q -O diar_infer_meeting.yaml https://raw.githubusercontent.com/NVIDIA-NeMo/Speech/main/examples/speaker_tasks/diarization/conf/inference/diar_infer_meeting.yaml

wav = Path("podcast_16k.wav").resolve()
Path("manifest.json").write_text(json.dumps({
    "audio_filepath": str(wav),
    "offset": 0, "duration": None, "label": "infer", "text": "-",
    "num_speakers": 2, "rttm_filepath": None, "uem_filepath": None,
}) + "\n")

cfg = OmegaConf.load("diar_infer_meeting.yaml")
cfg.num_workers = 1
cfg.diarizer.manifest_filepath = "manifest.json"
cfg.diarizer.out_dir = "nemo_out"
cfg.diarizer.oracle_vad = False
cfg.diarizer.vad.model_path = "vad_multilingual_marblenet"
cfg.diarizer.speaker_embeddings.model_path = "titanet_large"
cfg.diarizer.speaker_embeddings.parameters.save_embeddings = False
cfg.diarizer.clustering.parameters.oracle_num_speakers = True

print("VAD", cfg.diarizer.vad.model_path)
print("EMB", cfg.diarizer.speaker_embeddings.model_path)

sd_model = ClusteringDiarizer(cfg=cfg)
if torch.cuda.is_available():
    sd_model = sd_model.to("cuda")
sd_model.diarize()

rttms = list(Path("nemo_out").rglob("*.rttm"))
print("RTTM", rttms)
print(Path(rttms[0]).read_text()[:600] if rttms else "NO RTTM")

In [ ]:
from pathlib import Path
import json, zipfile, shutil
import numpy as np
import soundfile as sf
from transformers import AutoModel

MERGE_GAP = 3.0
MIN_SEC = 0.8
ASR_CHUNK = 24.0

wav_path = Path("podcast_hq.wav")
rttm_path = Path("nemo_out/pred_rttms/podcast_16k.rttm")
assert wav_path.exists() and rttm_path.exists()

def norm(label):
    d = "".join(c for c in str(label) if c.isdigit())
    return f"SPEAKER_{int(d):02d}" if d else str(label)

turns = []
for line in rttm_path.read_text().splitlines():
    p = line.split()
    if len(p) < 8 or p[0] != "SPEAKER":
        continue
    s, dur = float(p[3]), float(p[4])
    turns.append({"start": s, "end": s + dur, "speaker": norm(p[7])})
turns.sort(key=lambda t: t["start"])

merged = []
for t in turns:
    if merged and t["speaker"] == merged[-1]["speaker"] and t["start"] - merged[-1]["end"] <= MERGE_GAP:
        merged[-1]["end"] = t["end"]
    else:
        merged.append(dict(t))
print("turns", len(turns), "merged", len(merged))

audio, sr = sf.read(str(wav_path), dtype="float32")
print("cut from", wav_path, "sr", sr)
assert sr == 44100, "ожидали 44.1 kHz из исходного Opus"

out = Path("nemo_speakers_native")
if out.exists():
    shutil.rmtree(out)
out.mkdir()

print("Loading GigaAM...")
asr = AutoModel.from_pretrained(
    "ai-sage/GigaAM-Multilingual", revision="large_ctc", trust_remote_code=True,
)

def asr_text(path: Path) -> str:
    a, s = sf.read(str(path), dtype="float32")
    hop = int(ASR_CHUNK * s)
    parts = []
    for i in range(0, max(len(a), 1), hop):
        piece = a[i:i + hop]
        if len(piece) < int(0.4 * s):
            continue
        tmp = path.with_name(f".tmp_{i}.wav")
        sf.write(str(tmp), piece, s)
        try:
            r = asr.transcribe(str(tmp))
            parts.append((getattr(r, "text", None) or str(r)).strip())
        except Exception as e:
            print("  warn", path.name, e)
        finally:
            tmp.unlink(missing_ok=True)
    return " ".join(p for p in parts if p)

clips, n = [], {}
for t in merged:
    start, end, speaker = t["start"], t["end"], t["speaker"]
    if end - start < MIN_SEC:
        continue
    clip = audio[int(start * sr):int(end * sr)]
    if len(clip) < int(MIN_SEC * sr):
        continue
    if float(np.sqrt(np.mean(clip ** 2))) < 0.003:
        continue
    n[speaker] = n.get(speaker, 0) + 1
    name = f"{n[speaker]:04d}_{start:08.2f}-{end:08.2f}.wav"
    dest = out / speaker / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(dest), clip, sr)
    text = asr_text(dest)
    clips.append({
        "audio": f"{speaker}/{name}", "speaker": speaker,
        "start_sec": round(start, 2), "end_sec": round(end, 2), "text": text,
    })
    print(f"{speaker} {n[speaker]:04d}  {start:.1f}-{end:.1f}s ({end-start:.1f}s) sr={sr}  {text[:60]}")

(out / "podcast.json").write_text(json.dumps({
    "podcast": "h1615IhITQ4",
    "language": "ky",
    "diarizer": "NVIDIA NeMo ClusteringDiarizer (vad_multilingual_marblenet + titanet_large)",
    "asr": "GigaAM-Multilingual large_ctc",
    "audio": "yt-dlp native opus, one decode to 44.1 kHz wav, no mp3, no denoise",
    "clips": clips,
}, ensure_ascii=False, indent=2), encoding="utf-8")

zip_path = Path("nemo_speakers_native.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(out.rglob("*")):
        if p.is_file() and p.suffix in {".wav", ".json"}:
            z.write(p, p.relative_to(out))
print("ZIP", zip_path, zip_path.stat().st_size, "speakers", n, "clips", len(clips))